This notebook runs quality control and preprocessing on the concatenated Dumitru et al. dataset: cell filtering, highly variable gene selection, PCA, batch correction, UMAP.

Input: .h5ad file of Dumitru et al. dataset across all donors in ../data/dumitru/single_nucleus/h5ad/dumitru_all_donors_raw_concat.h5ad  
Output: preprocessed .h5ad file in ../data/dumitru/single_nucleus/h5ad/dumitru_all_donors_pp.h5ad

In [ ]:
import scanpy as sc

In [ ]:
adata = sc.read_h5ad('../data/dumitru/single_nucleus/h5ad/dumitru_all_donors_raw_concat.h5ad')
adata

In [ ]:
# QC filtering
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
sc.pp.filter_cells(adata, min_genes=200)
adata = adata[adata.obs.n_genes_by_counts < 15000, :].copy()
adata = adata[adata.obs.pct_counts_mt < 20, :].copy()
adata

In [ ]:
# Saving count data
adata.layers["counts"] = adata.X.copy()

# Normalize and log-transform the data 
sc.pp.normalize_total(adata, target_sum=1e4) 
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=5000, batch_key='patient_id', subset=False)

markers = ['ABCG2','ACVR1','ACVR1B','ACVR1C','ACVR2A','ACVR2B','ACVRL1','ADAM10','ADAM12','AIF1','ALDH1L1','AMH','AMHR2','APH1A','APH1B','APC','APC2','APOLD1','AQP4','ASCL1','ASPM','ATF2','AXIN1','AXIN2','BASP1','BASP1-AS1','BASP1P1','BCL9','BCL9L','BMP1','BMP10','BMP15','BMP2','BMP2K','BMP2KL','BMP3','BMP4','BMP5','BMP6','BMP6P1','BMP7','BMP7-AS1','BMP8A','BMP8B','BMP8B-AS1','BMPER','BMPR1A','BMPR1AP1','BMPR1AP2','BMPR1B','BMPR1B-DT','BMPR2','BTRC','BTG2','BUB1','BUB1B','CALB2','CAMK2A','CAV1','CBL','CCND1','CCND2','CCND3','CD133','CD15','CDCP1','CDH2','CER1','CFAP54','CHRLD1','CLDN5','CNTN1','CREBBP','CSNK1A1','CSNK1E','CTBP1','CTBP2','CTNNB1','CTNNBIP1','CX3CR1','CXCR4','DCX','DLL1','DLL3','DLL4','DVL1','DVL2','DVL3','EGFR','ELAVL2','EOMES','EPHA3','EPHA5','ETNPPL','EZH2','FABP7','FLT1','FOXJ1','FZD1','FZD10','FZD2','FZD3','FZD4','FZD5','FZD9','FXYD7','GAD1','GAD2','GFAP','GLAST','GLRA2','GRM3','GSK3A','GSK3B','HES1','HES5','HES6','HMGB2','HOPX','ID4','MBP','MOBP','MSI1','MSI2','NES','NEUROD1','NOG','NOTCH1','NOTCH2','NOTCH2NLA','NOTCH2NLB','NOTCH2NLC','NOTCH2NLR','NOTCH2P1','NOTCH3','NOTCH4','NPY','NR2E1','NRGN','NTF3','OLIG2','PAX3','PAX6','PDGFRA','PDGFRB','PECAM1','PKCZ','PLP1','PPIH','PROM1','PROX1','PTBP1','PVALB','P2RY12','RARRES2','RARRES2P1','RARRES2P2','RARRES2P3','RARRES2P4','RARRES2P6','RARRES2P8','RBFOX3','RELN','REST','RHCG','ROBO1','ROR2','RRM2','RSPH1','S100B','SATB2','SLC1A2','SLC1A3','SLC17A7','SLC2A1','SMARCA4','SMC4','SOX1','SOX10','SOX11','SOX2','SOX4','SOX5','SOX9','SSEA1','SST','ST8SIA2','SULF1','TBR1','TFAP2C','THBS4','THY1','TMEM119','TNC','VIP','VIM','VWF','WT1']
cell_cycle_genes = [x.strip() for x in open('../Data/regev_lab_cell_cycle_genes.txt')]

# Subset to HVGs and neurogenic markers
keep = adata.var['highly_variable'] | adata.var_names.isin(markers) | adata.var_names.isin(cell_cycle_genes) | adata.var_names.isin(genes)
adata = adata[:, keep].copy()
adata

In [ ]:
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])

In [ ]:
import scanpy.external as sce

# Harmony batch correction (creates X_pca_harmony)
sc.tl.pca(adata)
sce.pp.harmony_integrate(adata, key="patient_id", max_iter_harmony=50)

# Neighbors / UMAP / clustering on Harmony PCs
sc.pp.neighbors(adata, use_rep="X_pca_harmony", n_pcs=30, n_neighbors=20)
sc.tl.umap(adata, min_dist=0.3)

In [ ]:
sc.pl.umap(adata, color=['patient_id'])

In [ ]:
adata.write_h5ad('../data/dumitru/single_nucleus/h5ad/dumitru_all_donors_pp.h5ad')